# TRACR Town05 Co-Simulation Attack Demo

This notebook uses the existing TRACR dashboard shell, but the simulation loop is now driven by `V2VCoSimClientMaster.step()`. That means CARLA vehicles are controlled by the project V2V path planner/controller stack, while Simu5G carries normal and injected BSM messages.

The demo loads `V2V-Attack-Dataset/scenarios/intersection_4way/intersection_4way_scenario_01.yaml` and launches six vehicles through the unsignalized Town05 intersection: south-to-north, south-to-east, north-to-east, west-to-south, east-to-south, and north-to-west. Vehicle 1 is the ego and obstacle ghost attack target. The fake obstacle is highlighted only in the dashboard bird-eye image as a small orange-red overlay, so it is not visible to the ego vehicle camera.

## Setup

Load the same TRACR support utilities used by `tracr_demo.ipynb`, plus the obstacle ghost attack hook.

In [1]:
from pathlib import Path
from types import SimpleNamespace
import importlib
import os
import sys
import time

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "tutorials":
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import carla
import tutorials.tracr_demo_support as tracr_demo_support
tracr_demo_support = importlib.reload(tracr_demo_support)

TRACRDashboard = tracr_demo_support.TRACRDashboard
CarlaSensorPanel = tracr_demo_support.CarlaSensorPanel

from clients.V2V_CoSimClient_master import V2VCoSimClientMaster
from cosim_utils.attack_manager import ObstacleGhostVehicleAttack, V2XAttackManager, assign_attack_vehicle_ids
from cosim_utils.helpers import is_port_open, load_scenario, set_random_seed
from utils.carla_util import destroy_carla_actor, open_carla
from utils.simu5g_v2x_util import start_simu5g_bridge_in_terminal
from utils.util import prepare_sim_dirs, read_run_config, run_simulation_in_docker

os.environ["OMNETPP_HOME"] = "/home/siwen/Software/omnetpp-6.1"
os.environ["INET_HOME"] = "/home/siwen/Software/inet-4.5.4"
os.environ["SIMU5G_HOME"] = "/home/siwen/Software/Simu5G"
os.environ["PATH"] = (
    f'{os.environ["OMNETPP_HOME"]}/bin:'
    f'{os.environ["INET_HOME"]}/src:'
    f'{os.environ["SIMU5G_HOME"]}/src:'
    + os.environ.get("PATH", "")
)

print("Repository root:", REPO_ROOT)

Repository root: /home/siwen/Software/METS-R_docker/METS-R_HPC


## Launch services

This cell starts/reuses Simu5G, CARLA, and METS-R, then creates a `V2VCoSimClientMaster`. The four trips are generated sequentially on `48 -> -3`, and the second vehicle is locked as the dashboard ego/attack target.

In [2]:
VEINS_PORT = 9099
SCENARIO_PATH = REPO_ROOT / "V2V-Attack-Dataset/scenarios/intersection_4way/intersection_4way_scenario_01.yaml"
MOVEMENT_NAMES = [
    "south_to_north",
    "south_to_east",
    "north_to_east",
    "west_to_south",
    "east_to_south",
    "north_to_west",
]
VEHICLE_IDS = [1, 2, 3, 4, 5, 6]
EGO_VEHICLE_ID = 1
TRIP_DEPARTURE_GAP_TICKS = 20
RANDOM_SEED = 42
ATTACK_START_TICK = 20
ATTACK_END_TICK = 280

scenario = load_scenario(SCENARIO_PATH)
COSIM_ROADS = list(scenario.get("cosim_roads") or [])
movements = scenario.get("movements") or {}
TRIP_SPECS = []
for vid, movement_name in zip(VEHICLE_IDS, MOVEMENT_NAMES):
    movement = movements[movement_name]
    TRIP_SPECS.append((vid, movement_name, movement["origin"], movement["destination"]))

config = read_run_config(scenario["config_file"])
set_random_seed(RANDOM_SEED, config=config)
config.display_all = False
config.verbose = False
config.enable_debug_draw = False
config.draw_route_plan = False
config.v2v_position_mode = "local"
config.cv2x_communication_range_m = 500.0
config.carla_tick_timeout = 5.0
config.metsr_tick_timeout = 5.0
config.release_queued_cosim_vehicles = True
config.metsr_road = COSIM_ROADS
config.controller_vids = VEHICLE_IDS
config.handoff_spawn_clearance_m = 10.0
if scenario.get("network_file"):
    config.network_file = scenario["network_file"]
if scenario.get("town"):
    config.carla_map = scenario["town"]

print(f"Loaded scenario {scenario.get('scenario_id', SCENARIO_PATH.name)} from {SCENARIO_PATH}")
print("Trip specs:", TRIP_SPECS)

if not is_port_open(VEINS_PORT):
    print(f"Starting Simu5G bridge on port {VEINS_PORT}...")
    start_simu5g_bridge_in_terminal(REPO_ROOT, wait_seconds=5.0)
else:
    print(f"Simu5G bridge already listening on port {VEINS_PORT}.")

prepare_sim_dirs(config)

carla_client, carla_tm = open_carla(config)
world = carla_client.get_world()
world.set_weather(carla.WeatherParameters.ClearNoon)
set_random_seed(RANDOM_SEED, traffic_manager=carla_tm)
print("CARLA connected.")

metsr_port = int(config.metsr_port[0] if hasattr(config, "metsr_port") else config.ports[0])
if not is_port_open(metsr_port):
    run_simulation_in_docker(config)
else:
    print(f"METS-R already listening on port {metsr_port}; reusing it.")

cosim_client = V2VCoSimClientMaster(
    config,
    carla_client,
    carla_tm,
    controller_vids=VEHICLE_IDS,
    require_simu5g_uu=True,
)

if (scenario.get("traffic_lights") or {}).get("enabled") is False:
    light_count = cosim_client.set_unsignalized_intersection_lights()
    print(f"Set {light_count} CARLA traffic lights to frozen yellow.")

camera_x, camera_y = scenario.get("intersection_center_xy_carla", [0.0, 0.0])
cosim_client.set_custom_camera(camera_x, camera_y, 120.0)

for index, (vid, movement_name, road_from, road_to) in enumerate(TRIP_SPECS):
    if index > 0 and TRIP_DEPARTURE_GAP_TICKS > 0:
        cosim_client.metsr.tick(TRIP_DEPARTURE_GAP_TICKS, max_wait_seconds=10, poll_timeout=1)
    print(f"Generating trip veh={vid} movement={movement_name}: {road_from} -> {road_to}")
    cosim_client.metsr.generate_trip_between_roads([vid], road_from, road_to)
    cosim_client.metsr.update_vehicle_sensor_type([vid], "cv2x", True)

attack = ObstacleGhostVehicleAttack(
    target_vehicle_id=EGO_VEHICLE_ID,
    ghost_id=None,
    start_tick=cosim_client.current_tick + ATTACK_START_TICK,
    end_tick=cosim_client.current_tick + ATTACK_END_TICK,
    attack_id="tracr_v2v_master_obstacle_ghost",
)
assign_attack_vehicle_ids(attack, VEHICLE_IDS)
attacks = V2XAttackManager([attack])

sensor_panel = CarlaSensorPanel(world, carla, destroy_carla_actor)
sensor_panel.spawn_overhead_camera(x=camera_x, y=camera_y, z=120.0)
sensor_panel.target_vehicle_id = EGO_VEHICLE_ID
sensor_panel.strict_target = True

runtime = SimpleNamespace(
    config=config,
    metsr=cosim_client.metsr,
    world=world,
    carla_client=carla_client,
    carla_tm=carla_tm,
    carla_state=SimpleNamespace(active_vehicles=cosim_client.carla_vehs, display_vehicles=cosim_client.displayOnly_vehs),
    sensor_panel=sensor_panel,
    generated_vehicle_ids=VEHICLE_IDS,
    v2x_vehicle_ids=VEHICLE_IDS,
    focus_vehicle_id=EGO_VEHICLE_ID,
    lock_focus_vehicle=True,
    bsm_stream_source="simu5g",
    bsm_stream_label="Simu5G + V2V master obstacle ghost attack",
)

{
    "scenario": str(SCENARIO_PATH),
    "cosim_roads": COSIM_ROADS,
    "trip_specs": TRIP_SPECS,
    "ego_vehicle_id": EGO_VEHICLE_ID,
    "attack": vars(attack),
}


Loaded scenario intersection_4way_scenario_01 from /home/siwen/Software/METS-R_docker/METS-R_HPC/V2V-Attack-Dataset/scenarios/intersection_4way/intersection_4way_scenario_01.yaml
Trip specs: [(1, 'south_to_north', '40', '47'), (2, 'south_to_east', '40', '0'), (3, 'north_to_east', '-47', '0'), (4, 'west_to_south', '1', '-40'), (5, 'east_to_south', '-0', '-40'), (6, 'north_to_west', '-47', '-1')]
Starting Simu5G bridge on port 9099...


Resizing viewport due to setres change, 1600 x 1000


CARLA connected.
Connection established!
Set 54 CARLA traffic lights to frozen yellow.
Generating trip veh=1 movement=south_to_north: 40 -> 47
Generating trip veh=2 movement=south_to_east: 40 -> 0
Generating trip veh=3 movement=north_to_east: -47 -> 0
Generating trip veh=4 movement=west_to_south: 1 -> -40
Generating trip veh=5 movement=east_to_south: -0 -> -40
Generating trip veh=6 movement=north_to_west: -47 -> -1


{'scenario': '/home/siwen/Software/METS-R_docker/METS-R_HPC/V2V-Attack-Dataset/scenarios/intersection_4way/intersection_4way_scenario_01.yaml',
 'cosim_roads': ['-47',
  '47',
  '-17',
  '17',
  '-0',
  '0',
  '-40',
  '40',
  '-18',
  '18',
  '-1',
  '1'],
 'trip_specs': [(1, 'south_to_north', '40', '47'),
  (2, 'south_to_east', '40', '0'),
  (3, 'north_to_east', '-47', '0'),
  (4, 'west_to_south', '1', '-40'),
  (5, 'east_to_south', '-0', '-40'),
  (6, 'north_to_west', '-47', '-1')],
 'ego_vehicle_id': 1,
 'attack': {'attack_id': 'tracr_v2v_master_obstacle_ghost',
  'start_tick': 120,
  'end_tick': 380,
  'enabled': True,
  'target_vehicle_id': 1,
  'ghost_id': 7,
  'lead_time_s': 0.0,
  'base_distance_m': 13.0,
  'ghost_speed_mps': 2.0}}

## Runtime diagnostics

Use this only when needed. It reports the METS-R queue/co-sim state and the CARLA-managed vehicle IDs from `V2VCoSimClientMaster`.

In [ ]:
print("tick:", cosim_client.current_tick)
print("generated:", VEHICLE_IDS)
print("ego:", EGO_VEHICLE_ID)
print("carla managed:", sorted(cosim_client.carla_vehs.keys()))
print("controllers:", sorted(cosim_client.controllers.keys()))
print("route synced:", dict(cosim_client.route_synced))

print()
print("query_vehicle:")
try:
    print(cosim_client.metsr.query_vehicle(
        id=VEHICLE_IDS,
        private_veh=[True] * len(VEHICLE_IDS),
        transform_coords=True,
    ))
except Exception as exc:
    print("query_vehicle failed:", exc)

print()
print("coSimVehicle:")
print(cosim_client.metsr.query_coSimVehicle())

print()
print("last v2x rows:", len(getattr(cosim_client, "last_v2x_rows", []) or []))
print("last attack injection vehicles:", attacks.last_injection.get("vehicles", []))

## Demo dashboard

Open the printed local dashboard URL. The BSM panel shows the ego vehicle's received Simu5G BSM stream, including the injected obstacle ghost BSM when it is delivered. CARLA also shows an orange-red live marker at the falsified ghost position ahead of ego.

In [ ]:
if not hasattr(runtime, "viz_info"):
    runtime.viz_info = tracr_demo_support._start_viz_with_port_fallback(cosim_client.metsr, {})

dashboard = TRACRDashboard(
    stream_url=runtime.viz_info["url"],
    fullscreen=True,
    bsm_stream_label=runtime.bsm_stream_label,
    bsm_ego_only=True,
    title="V2V Attack Co-Simulation Demo",
)
dashboard_url = dashboard.display_external(port=8899)
dashboard_url

METS-R Vis live stream is available at ws://127.0.0.1:8765; origin=(0.0, 0.0); call render() to send frames.
Serving /home/siwen/Software/METS-R_docker/METS-R_HPC/output/tracr_dashboard with CORS enabled on port 8899...


'http://127.0.0.1:8899/index.html'

127.0.0.1 - - [05/Jul/2026 11:22:03] "GET /index.html HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:22:03] "GET /state.json?ts=1783264923463 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:22:03] "GET /state.json?ts=1783264923966 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:22:04] "GET /state.json?ts=1783264924466 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:22:04] "GET /state.json?ts=1783264924966 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:22:05] "GET /state.json?ts=1783264925466 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:22:05] "GET /state.json?ts=1783264925966 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:22:06] "GET /state.json?ts=1783264926466 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:22:06] "GET /state.json?ts=1783264926966 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:22:07] "GET /state.json?ts=1783264927466 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:22:07] "GET /state.json?ts=1783264927966 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:22:08] "GET /state.json?ts=178

## Run the live attack loop

This loop advances the real V2V co-simulation stack with `V2VCoSimClientMaster.step()`, updates the dashboard sensors for the locked ego vehicle, draws the obstacle ghost only on the dashboard bird-eye image as a small orange-red overlay, and refreshes the TRACR dashboard display.

In [4]:
def _dashboard_state():
    return SimpleNamespace(
        active_vehicles=cosim_client.carla_vehs,
        display_vehicles=cosim_client.displayOnly_vehs,
    )


def _draw_obstacle_ghost_box():
    markers = []
    for vehicle in attacks.last_injection.get("vehicles", []) or []:
        if vehicle.get("role") != "obstacle_ghost_attacker":
            continue
        try:
            location = cosim_client.get_carla_location(vehicle["x"], vehicle["y"])
            location.z += 1.0
            markers.append({"location": location, "size_px": 14})
        except Exception:
            pass
    runtime.sensor_panel.set_overhead_markers(markers)


def run_v2v_master_demo(ticks=1500, sleep_s=0.0, dashboard_every=3, render_every=2):
    last = None
    for step_index in range(int(ticks)):
        predicted_tick = cosim_client.current_tick + 1
        attack_active = any(item.active(predicted_tick) for item in attacks.attacks)
        phase = "obstacle_ghost_attack" if attack_active else "normal"
        step_result = cosim_client.step(extra_v2x_messages=attacks, phase=phase)
        runtime.carla_state = _dashboard_state()
        runtime.sensor_panel.ensure_sensors(runtime.carla_state, preferred_vehicle_ids=[EGO_VEHICLE_ID])
        _draw_obstacle_ghost_box()

        bsm_records = list(step_result.get("v2x", {}).get("stream", []) or [])
        dashboard_step = {
            "state": runtime.carla_state,
            "v2x": step_result.get("v2x", {}),
            "tracr_projection": {"focus_vehicle": EGO_VEHICLE_ID},
        }
        render_info = None
        render_error = None
        if render_every <= 1 or step_index % int(render_every) == 0:
            try:
                render_info = cosim_client.metsr.render(client_wait_timeout=0)
            except Exception as exc:
                render_error = str(exc).splitlines()[0]
        if dashboard is not None and (dashboard_every <= 1 or step_index % int(dashboard_every) == 0):
            dashboard.update(runtime, dashboard_step, bsm_records, render_info=render_info, render_error=render_error)
        last = {
            "tick": step_result.get("tick"),
            "phase": phase,
            "carla_vehicles": sorted(cosim_client.carla_vehs.keys()),
            "controllers": sorted(cosim_client.controllers.keys()),
            "route_synced": dict(cosim_client.route_synced),
            "bsm_count": len(bsm_records),
            "last_attack_vehicles": attacks.last_injection.get("vehicles", []),
        }
        if sleep_s:
            time.sleep(float(sleep_s))
    return last

last_result = run_v2v_master_demo(ticks=1500, sleep_s=0.0, dashboard_every=3, render_every=2)
last_result

127.0.0.1 - - [05/Jul/2026 11:23:03] "GET /state.json?ts=1783264983399 HTTP/1.1" 200 -


[handoff] veh=3 metsr_speed=0.00 spawn_loc=(-51.40,-73.66,0.50) spawn_yaw=89.66 target_velocity=(0.06,10.00)
Vehicle 3 entered the co-sim ownership set and is now CARLA-managed.
[handoff] veh=5 metsr_speed=0.00 spawn_loc=(15.52,-0.98,0.50) spawn_yaw=179.86 target_velocity=(-10.00,0.02)
Vehicle 5 entered the co-sim ownership set and is now CARLA-managed.
[handoff] veh=1 metsr_speed=0.00 spawn_loc=(-43.82,73.73,0.50) spawn_yaw=-89.62 target_velocity=(0.07,-10.00)
Vehicle 1 entered the co-sim ownership set and is now CARLA-managed.
[handoff] veh=4 metsr_speed=0.00 spawn_loc=(-107.75,6.32,0.50) spawn_yaw=-0.15 target_velocity=(10.00,-0.03)
Vehicle 4 entered the co-sim ownership set and is now CARLA-managed.


127.0.0.1 - - [05/Jul/2026 11:23:03] "GET /state.json?ts=1783264983899 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:23:04] "GET /state.json?ts=1783264984399 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:23:04] "GET /state.json?ts=1783264984899 HTTP/1.1" 200 -


Vehicle 6 handoff delayed because vehicle 3 is still within 10.0 m of (-54.90,-73.64).
Vehicle 2 handoff delayed because vehicle 1 is still within 10.0 m of (-43.82,73.73).


127.0.0.1 - - [05/Jul/2026 11:23:05] "GET /state.json?ts=1783264985399 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:23:05] "GET /state.json?ts=1783264985899 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:23:06] "GET /state.json?ts=1783264986399 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:23:06] "GET /state.json?ts=1783264986899 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:23:07] "GET /state.json?ts=1783264987399 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:23:07] "GET /state.json?ts=1783264987899 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:23:08] "GET /state.json?ts=1783264988399 HTTP/1.1" 200 -


[handoff] veh=6 metsr_speed=0.00 spawn_loc=(-54.90,-73.64,0.50) spawn_yaw=89.66 target_velocity=(0.06,10.00)
Vehicle 6 entered the co-sim ownership set and is now CARLA-managed.


127.0.0.1 - - [05/Jul/2026 11:23:08] "GET /state.json?ts=1783264988899 HTTP/1.1" 200 -


[handoff] veh=2 metsr_speed=0.00 spawn_loc=(-43.82,73.73,0.50) spawn_yaw=-89.62 target_velocity=(0.07,-10.00)
Vehicle 2 entered the co-sim ownership set and is now CARLA-managed.


127.0.0.1 - - [05/Jul/2026 11:23:09] "GET /state.json?ts=1783264989399 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:23:09] "GET /state.json?ts=1783264989899 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:23:10] "GET /state.json?ts=1783264990399 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:23:10] "GET /state.json?ts=1783264990899 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:23:11] "GET /state.json?ts=1783264991399 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:23:11] "GET /state.json?ts=1783264991899 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:23:12] "GET /state.json?ts=1783264992399 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:23:12] "GET /state.json?ts=1783264992899 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:23:13] "GET /state.json?ts=1783264993399 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:23:13] "GET /state.json?ts=1783264993899 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:23:14] "GET /state.json?ts=1783264994399 HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 11:23:14] "GET /

KeyboardInterrupt: 

## Manual step mode

Use this cell for a controlled walkthrough. Each call advances one `V2VCoSimClientMaster.step()` and refreshes the dashboard once.

In [ ]:
predicted_tick = cosim_client.current_tick + 1
attack_active = any(item.active(predicted_tick) for item in attacks.attacks)
phase = "obstacle_ghost_attack" if attack_active else "normal"
step_result = cosim_client.step(extra_v2x_messages=attacks, phase=phase)
runtime.carla_state = _dashboard_state()
runtime.sensor_panel.ensure_sensors(runtime.carla_state, preferred_vehicle_ids=[EGO_VEHICLE_ID])
_draw_obstacle_ghost_box()
bsm_records = list(step_result.get("v2x", {}).get("stream", []) or [])
dashboard.update(
    runtime,
    {"state": runtime.carla_state, "v2x": step_result.get("v2x", {}), "tracr_projection": {"focus_vehicle": EGO_VEHICLE_ID}},
    bsm_records,
    render_info=cosim_client.metsr.render(client_wait_timeout=0),
)
{
    "tick": step_result.get("tick"),
    "phase": phase,
    "carla_vehicles": sorted(cosim_client.carla_vehs.keys()),
    "controllers": sorted(cosim_client.controllers.keys()),
    "route_synced": dict(cosim_client.route_synced),
    "bsm_count": len(bsm_records),
    "last_attack_vehicles": attacks.last_injection.get("vehicles", []),
}

## Cleanup

Run this when the demo is done.

In [5]:
try:
    sensor_panel.close()
except Exception:
    pass
try:
    cosim_client.close()
except Exception as exc:
    print("cleanup warning:", exc)

FUnixPlatformMisc::RequestExitWithStatus
 FUnixPlatformMisc::RequestExit
 FUnixPlatformMisc::RequestExit
 

Terminated
